In [ ]:
%env AG_DATA=/content/ag_data
!mkdir -p $AG_DATA/ref $AG_DATA/weights
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

env: AG_DATA=/content/ag_data
name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07


In [ ]:
# torch runtime (the port); no jax here. clone once, pull after every local push
!git -C /content/ag pull -q 2>/dev/null || git clone -q -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag
!pip install -q -e /content/ag pyfaidx
%cd /content/ag/experiments
!git log -1 --oneline

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for alphagenome-pytorch (pyproject.toml) ... done
/content/ag/experiments
43b19ea (HEAD -> experiments-profiling, origin/experiments-profiling) phase 2: module tree + perfetto trace (port, 131k)


In [ ]:
# same reference genome + index as the golden run
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa
!test -s $AG_DATA/ref/GRCh38.p13.genome.fa.fai || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa.fai https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa.fai
!ls -lh $AG_DATA/ref

total 3.1G
-rw-r--r-- 1 root root 3.1G Dec 18  2025 GRCh38.p13.genome.fa
-rw-r--r-- 1 root root  22K Dec 18  2025 GRCh38.p13.genome.fa.fai


In [ ]:
# step 1: golden input rebuilt with the port, must match both hashes
!python scripts/port_input.py -v

chr19:40939304-41070376 (131072, 4) hashes match


In [ ]:
# step 2: pass 1, module tree with shapes -> results/modules_131k.txt
!python scripts/trace_modules.py --check -v


model_all_folds.safetensors: downloading bytes:  23% 212M/921M [00:01<00:02, 306MB/s, 12.9MB/s  ]  
model_all_folds.safetensors: downloading bytes:  80% 738M/921M [00:02<00:00, 476MB/s, 60.6MB/s  ]
model_all_folds.safetensors: downloading bytes:  86% 790M/921M [00:03<00:00, 161MB/s, 65.3MB/s  ] ]
model_all_folds.safetensors: downloading bytes: 100% 813M/813M [00:06<00:00, 132MB/s, 57.0MB/s  ] ]
model_all_folds.safetensors: reconstructing file: 100% 921M/921M [00:06<00:00, 150MB/s, 71.3MB/s  ]
/content/ag/experiments/results/modules_131k.txt 439 module calls


In [ ]:
# step 3: pass 2, perfetto trace -> results/trace_131k.json
!python scripts/trace_kernels.py -v

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/content/ag/experiments/results/trace_131k.json
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   root         0.00%       0.000us         0.00% 

In [ ]:
!ls -lh results
!head -30 results/modules_131k.txt

total 9.8M
-rw-r--r-- 1 root root  46K Oct 16 13:22 modules_131k.txt
-rw-r--r-- 1 root root 9.8M Oct 16 13:22 trace_131k.json
# input chr19:40939304-41070376 length 131072 policy DtypePolicy(params=float32, compute=bfloat16, output=bfloat16) gpu NVIDIA A100-SXM4-80GB torch 2.11.0+cu130
# fields: name type call params in out
root	AlphaGenome	1	450452613	[float32[1,131072,4],int64[1]]	{atac:{1:bfloat16[1,131072,256],128:bfloat16[1,1024,256]},dnase:{1:bfloat16[1,131072,384],128:bfloat16[1,1024,384]},procap:{1:bfloat16[1,131072,128],128:bfloat16[1,1024,128]},cage:{1:bfloat16[1,131072,640],128:bfloat16[1,1024,640]},rna_seq:{1:bfloat16[1,131072,768],128:bfloat16[1,1024,768]},chip_tf:{128:bfloat16[1,1024,1664]},chip_histone:{128:bfloat16[1,1024,1152]},contact_maps:bfloat16[1,64,64,28],splice_sites:{logits:bfloat16[1,131072,5],probs:bfloat16[1,131072,5]},splice_site_usage:{logits:bfloat16[1,131072,734],predictions:bfloat16[1,131072,734],track_mask:bfloat16[1,1,734]},splice_junctions:{pred_coun

In [8]:
!cd results && tar czf /tmp/results.tgz modules_131k.txt trace_131k.json && python -c "import base64; print('=====B64====='); print(base64.b64encode(open('/tmp/results.tgz','rb').read()).decode()); print('=====END=====')"

=====B64=====
...
=====END=====
